# Database Setup (Aligned to Actual Pipeline)

This notebook defines the PostgreSQL tables used by the **actual current pipeline source files**:

**Cleaning → FE → MSTL → LightGBM → MinT → C2G → Prescription → Finalize APIs**

It replaces the older warehouse-only setup with the real staging, mart, and `api_*` serving tables that the current code reads and writes.


## Confirmed table flow from the uploaded original source files

### Reads / writes by stage

- **Cleaning**
  - writes: `stg_cleaned_sales_panel`
- **FE**
  - reads: `stg_cleaned_sales_panel`
  - writes: `stg_feature_engineered_panel`, `stg_feature_engineered_panel_meta`, `stg_mstl_product_month`
- **MSTL**
  - reads: `stg_mstl_product_month`
  - writes: `stg_mstl_decomposition_for_lightgbm`, `stg_mstl_seasonal_profiles`
  - also writes API tables: `api_overview_kpis`, `api_overview_trend_chart`, `api_seasonality_monthly`, `api_seasonality_insights`, `api_trend_true_growth_table`, `api_trend_true_growth_summary`
- **LightGBM**
  - reads: `stg_feature_engineered_panel`, `stg_mstl_decomposition_for_lightgbm`, `stg_mstl_seasonal_profiles`
  - writes: `stg_lightgbm_base_forecasts`, `stg_lightgbm_dashboard_forecast`, `stg_lightgbm_forecast_metrics`, `stg_lightgbm_hierarchy_metrics_val`, `stg_lightgbm_hierarchy_metrics_test`
  - also writes API tables: `api_forecast_metrics`, `api_forecast_insights`
- **MinT**
  - reads: `stg_feature_engineered_panel`, `stg_lightgbm_base_forecasts`
  - writes: `stg_mint_reconciled_forecast`, `stg_mint_coherence_checks`, `stg_mint_dashboard_forecast`
  - also writes API tables: `api_forecast_horizon`, `api_forecast_chart`, `api_coherence_summary`, `api_coherence_methodology`, `api_coherence_failed_checks`
- **C2G**
  - reads: `stg_mint_reconciled_forecast`, `stg_feature_engineered_panel`
  - writes: `mart_c2g_bottom_level`, `mart_c2g_all_levels`, `mart_c2g_top_gainers`, `mart_c2g_top_decliners`
  - also writes API tables: `api_c2g_top_contributors`, `api_c2g_drilldown`, `api_overview_growth_drivers`
- **Prescription**
  - reads: `stg_feature_engineered_panel`, `stg_mint_reconciled_forecast`, `mart_c2g_bottom_level`, `stg_mstl_decomposition_for_lightgbm`
  - optionally references: `stg_lightgbm_dashboard_forecast`
  - writes: `api_prescription_actions`, `api_prescription_implementation_guide`, `mart_prescription_expand`, `mart_prescription_maintain`, `mart_prescription_deprioritize`
- **Finalize_APIs**
  - reads: `api_overview_kpis`, `api_forecast_horizon`
  - validates all final API tables

### Important note
This setup is aligned to the current codebase. Some old tables from the previous notebook (for example `fact_decomposition`, `fact_forecast`, `fact_allocation`) are **not part of the current pipeline path anymore**.


In [ ]:
# Configuration
from sqlalchemy import create_engine, text

DB_USER = "postgres"
DB_PASSWORD = "Nestle123"
DB_HOST = "127.0.0.1"
DB_PORT = "5432"
DB_NAME = "nestle_forecasting"
SCHEMA = "retail"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    future=True,
    pool_pre_ping=True,
)

def run_sql(sql: str):
    with engine.begin() as conn:
        conn.execute(text(sql))


## 1) Create schema

In [ ]:
run_sql(f'CREATE SCHEMA IF NOT EXISTS "{SCHEMA}"')
print(f"Schema ready: {SCHEMA}")

## 2) Core dimension tables

These remain useful for hierarchy management and lookup consistency, even though the current pipeline mainly reads and writes denormalized stage/API tables.


In [ ]:

run_sql(f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."dim_region" (
    region_id SERIAL PRIMARY KEY,
    nestle_region TEXT UNIQUE NOT NULL
)
''')

run_sql(f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."dim_cluster" (
    cluster_id SERIAL PRIMARY KEY,
    nestle_store_cluster TEXT UNIQUE NOT NULL,
    nestle_region TEXT
)
''')

run_sql(f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."dim_store" (
    store_id SERIAL PRIMARY KEY,
    store_code TEXT UNIQUE NOT NULL,
    store_description TEXT,
    nestle_store_cluster TEXT,
    nestle_region TEXT
)
''')

run_sql(f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."dim_product" (
    product_id SERIAL PRIMARY KEY,
    product_code TEXT UNIQUE NOT NULL,
    product_description TEXT,
    category TEXT,
    brand TEXT
)
''')

run_sql(f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."dim_month" (
    ds DATE PRIMARY KEY,
    year INTEGER,
    quarter INTEGER,
    month INTEGER
)
''')

print("Dimension tables ready")


## 3) Stage tables used by the pipeline

These are the real handoff tables between notebooks/scripts.


In [ ]:

stage_sql = [

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."stg_cleaned_sales_panel" (
    ds DATE,
    product_code TEXT,
    product_description TEXT,
    store_code TEXT,
    store_description TEXT,
    nestle_store_cluster TEXT,
    nestle_region TEXT,
    category TEXT,
    brand TEXT,
    units_sold_ty DOUBLE PRECISION,
    units_sold_ly DOUBLE PRECISION,
    net_sales_ty DOUBLE PRECISION,
    net_sales_ly DOUBLE PRECISION,
    source_file TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."stg_feature_engineered_panel" (
    ds DATE,
    split TEXT,
    product_code TEXT,
    product_description TEXT,
    store_code TEXT,
    store_description TEXT,
    nestle_store_cluster TEXT,
    nestle_region TEXT,
    category TEXT,
    brand TEXT,
    month INTEGER,
    quarter INTEGER,
    year INTEGER,
    month_sin DOUBLE PRECISION,
    month_cos DOUBLE PRECISION,
    net_sales_ty DOUBLE PRECISION,
    lag_1 DOUBLE PRECISION,
    lag_2 DOUBLE PRECISION,
    lag_3 DOUBLE PRECISION,
    lag_6 DOUBLE PRECISION,
    lag_12 DOUBLE PRECISION,
    rolling_mean_3 DOUBLE PRECISION,
    rolling_mean_6 DOUBLE PRECISION,
    rolling_mean_12 DOUBLE PRECISION,
    rolling_std_3 DOUBLE PRECISION,
    rolling_std_6 DOUBLE PRECISION,
    rolling_std_12 DOUBLE PRECISION,
    trend_growth DOUBLE PRECISION,
    rolling_growth_3 DOUBLE PRECISION,
    is_observed_month BOOLEAN,
    is_gap_filled BOOLEAN,
    history_months_train INTEGER,
    nonzero_months_train INTEGER,
    demand_share_train DOUBLE PRECISION,
    total_sales_train DOUBLE PRECISION,
    filled_gap_share_train DOUBLE PRECISION,
    eligible_for_model BOOLEAN,
    eligible_for_strict_eval BOOLEAN,
    eligibility_tier TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."stg_feature_engineered_panel_meta" (
    metric TEXT,
    value TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."stg_mstl_product_month" (
    ds DATE,
    product_code TEXT,
    product_description TEXT,
    category TEXT,
    brand TEXT,
    net_sales_ty DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."stg_mstl_decomposition_for_lightgbm" (
    ds DATE,
    product_code TEXT,
    trend DOUBLE PRECISION,
    seasonal_strength DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."stg_mstl_seasonal_profiles" (
    product_code TEXT,
    month INTEGER,
    seasonal_multiplier DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."stg_lightgbm_base_forecasts" (
    ds DATE,
    split TEXT,
    product_code TEXT,
    store_code TEXT,
    nestle_store_cluster TEXT,
    nestle_region TEXT,
    store_description TEXT,
    product_description TEXT,
    brand TEXT,
    category TEXT,
    actual DOUBLE PRECISION,
    base_forecast DOUBLE PRECISION,
    fallback_forecast DOUBLE PRECISION,
    eligible_for_model BOOLEAN,
    eligible_for_strict_eval BOOLEAN
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."stg_lightgbm_dashboard_forecast" (
    ds DATE,
    product_code TEXT,
    store_code TEXT,
    nestle_store_cluster TEXT,
    nestle_region TEXT,
    store_description TEXT,
    product_description TEXT,
    brand TEXT,
    category TEXT,
    base_forecast DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."stg_lightgbm_forecast_metrics" (
    split TEXT,
    rmse DOUBLE PRECISION,
    mae DOUBLE PRECISION,
    smape DOUBLE PRECISION,
    mase DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."stg_lightgbm_hierarchy_metrics_val" (
    level TEXT,
    rmse DOUBLE PRECISION,
    mae DOUBLE PRECISION,
    smape DOUBLE PRECISION,
    mase DOUBLE PRECISION,
    split TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."stg_lightgbm_hierarchy_metrics_test" (
    level TEXT,
    rmse DOUBLE PRECISION,
    mae DOUBLE PRECISION,
    smape DOUBLE PRECISION,
    mase DOUBLE PRECISION,
    split TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."stg_mint_reconciled_forecast" (
    ds DATE,
    product_code TEXT,
    store_code TEXT,
    nestle_store_cluster TEXT,
    nestle_region TEXT,
    store_description TEXT,
    product_description TEXT,
    brand TEXT,
    category TEXT,
    actual DOUBLE PRECISION,
    base_forecast DOUBLE PRECISION,
    reconciled_forecast DOUBLE PRECISION,
    split TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."stg_mint_coherence_checks" (
    ds DATE,
    check_name TEXT,
    level TEXT,
    status TEXT,
    actual_value DOUBLE PRECISION,
    expected_value DOUBLE PRECISION,
    difference DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."stg_mint_dashboard_forecast" (
    ds DATE,
    level TEXT,
    nestle_region TEXT,
    nestle_store_cluster TEXT,
    product_code TEXT,
    store_code TEXT,
    actual DOUBLE PRECISION,
    base_forecast DOUBLE PRECISION,
    reconciled_forecast DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."mart_c2g_bottom_level" (
    ds DATE,
    level TEXT,
    nestle_region TEXT,
    nestle_store_cluster TEXT,
    store_code TEXT,
    store_description TEXT,
    product_code TEXT,
    product_description TEXT,
    category TEXT,
    brand TEXT,
    eligible_for_model BOOLEAN,
    reconciled_forecast DOUBLE PRECISION,
    actual_ly DOUBLE PRECISION,
    actual_last DOUBLE PRECISION,
    growth_vs_ly_abs DOUBLE PRECISION,
    growth_vs_last_abs DOUBLE PRECISION,
    growth_vs_ly_pct DOUBLE PRECISION,
    growth_vs_last_pct DOUBLE PRECISION,
    national_growth_vs_ly_abs DOUBLE PRECISION,
    national_growth_vs_last_abs DOUBLE PRECISION,
    c2g_vs_ly DOUBLE PRECISION,
    c2g_vs_last DOUBLE PRECISION,
    driver_direction TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."mart_c2g_all_levels" (
    ds DATE,
    level TEXT,
    nestle_region TEXT,
    nestle_store_cluster TEXT,
    store_code TEXT,
    store_description TEXT,
    product_code TEXT,
    product_description TEXT,
    category TEXT,
    brand TEXT,
    eligible_for_model BOOLEAN,
    reconciled_forecast DOUBLE PRECISION,
    actual_ly DOUBLE PRECISION,
    actual_last DOUBLE PRECISION,
    growth_vs_ly_abs DOUBLE PRECISION,
    growth_vs_last_abs DOUBLE PRECISION,
    growth_vs_ly_pct DOUBLE PRECISION,
    growth_vs_last_pct DOUBLE PRECISION,
    national_growth_vs_ly_abs DOUBLE PRECISION,
    national_growth_vs_last_abs DOUBLE PRECISION,
    c2g_vs_ly DOUBLE PRECISION,
    c2g_vs_last DOUBLE PRECISION,
    driver_direction TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."mart_c2g_top_gainers" (
    ds DATE,
    level TEXT,
    nestle_region TEXT,
    nestle_store_cluster TEXT,
    store_code TEXT,
    product_code TEXT,
    growth_vs_ly_abs DOUBLE PRECISION,
    c2g_vs_ly DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."mart_c2g_top_decliners" (
    ds DATE,
    level TEXT,
    nestle_region TEXT,
    nestle_store_cluster TEXT,
    store_code TEXT,
    product_code TEXT,
    growth_vs_ly_abs DOUBLE PRECISION,
    c2g_vs_ly DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."mart_prescription_expand" (
    ds DATE,
    level TEXT,
    nestle_region TEXT,
    nestle_store_cluster TEXT,
    product_code TEXT,
    recommendation TEXT,
    rationale TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."mart_prescription_maintain" (
    ds DATE,
    level TEXT,
    nestle_region TEXT,
    nestle_store_cluster TEXT,
    product_code TEXT,
    recommendation TEXT,
    rationale TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."mart_prescription_deprioritize" (
    ds DATE,
    level TEXT,
    nestle_region TEXT,
    nestle_store_cluster TEXT,
    product_code TEXT,
    recommendation TEXT,
    rationale TEXT
)
''',
]

for sql in stage_sql:
    run_sql(sql)

print("Stage / mart tables ready")


## 4) Final API tables used by the dashboard

These are the tables your frontend should ultimately consume.


In [ ]:

api_sql = [

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_overview_kpis" (
    as_of_month DATE,
    total_sales DOUBLE PRECISION,
    mom_growth DOUBLE PRECISION,
    ytd_growth DOUBLE PRECISION,
    next_3m_forecast DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_overview_trend_chart" (
    ds DATE,
    actual_sales DOUBLE PRECISION,
    trend_sales DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_overview_growth_drivers" (
    ds DATE,
    level TEXT,
    nestle_region TEXT,
    nestle_store_cluster TEXT,
    store_code TEXT,
    product_code TEXT,
    growth_vs_ly_abs DOUBLE PRECISION,
    driver_direction TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_seasonality_monthly" (
    month INTEGER,
    seasonal_multiplier DOUBLE PRECISION,
    seasonal_index DOUBLE PRECISION,
    category TEXT,
    brand TEXT,
    product_code TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_seasonality_insights" (
    sort_order INTEGER,
    label TEXT,
    value TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_trend_true_growth_table" (
    ds DATE,
    product_code TEXT,
    product_description TEXT,
    category TEXT,
    brand TEXT,
    trend_sales DOUBLE PRECISION,
    mom_trend_growth DOUBLE PRECISION,
    ytd_trend_growth DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_trend_true_growth_summary" (
    as_of_month DATE,
    avg_mom_trend_growth DOUBLE PRECISION,
    avg_ytd_trend_growth DOUBLE PRECISION,
    improving_series_count INTEGER,
    declining_series_count INTEGER
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_forecast_metrics" (
    split TEXT,
    rmse DOUBLE PRECISION,
    mae DOUBLE PRECISION,
    smape DOUBLE PRECISION,
    mase DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_forecast_chart" (
    ds DATE,
    level TEXT,
    actual DOUBLE PRECISION,
    base_forecast DOUBLE PRECISION,
    reconciled_forecast DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_forecast_horizon" (
    ds DATE,
    reconciled_forecast DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_forecast_insights" (
    sort_order INTEGER,
    label TEXT,
    value TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_coherence_summary" (
    check_group TEXT,
    passed_checks INTEGER,
    failed_checks INTEGER,
    total_checks INTEGER,
    pass_rate DOUBLE PRECISION
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_coherence_methodology" (
    sort_order INTEGER,
    label TEXT,
    value TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_coherence_failed_checks" (
    ds DATE,
    check_name TEXT,
    level TEXT,
    actual_value DOUBLE PRECISION,
    expected_value DOUBLE PRECISION,
    difference DOUBLE PRECISION,
    status TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_c2g_top_contributors" (
    ds DATE,
    level TEXT,
    nestle_region TEXT,
    nestle_store_cluster TEXT,
    store_code TEXT,
    store_description TEXT,
    product_code TEXT,
    product_description TEXT,
    category TEXT,
    brand TEXT,
    eligible_for_model BOOLEAN,
    reconciled_forecast DOUBLE PRECISION,
    actual_ly DOUBLE PRECISION,
    actual_last DOUBLE PRECISION,
    growth_vs_ly_abs DOUBLE PRECISION,
    growth_vs_last_abs DOUBLE PRECISION,
    growth_vs_ly_pct DOUBLE PRECISION,
    growth_vs_last_pct DOUBLE PRECISION,
    national_growth_vs_ly_abs DOUBLE PRECISION,
    national_growth_vs_last_abs DOUBLE PRECISION,
    c2g_vs_ly DOUBLE PRECISION,
    c2g_vs_last DOUBLE PRECISION,
    driver_direction TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_c2g_drilldown" (
    ds DATE,
    level TEXT,
    nestle_region TEXT,
    nestle_store_cluster TEXT,
    store_code TEXT,
    store_description TEXT,
    product_code TEXT,
    product_description TEXT,
    category TEXT,
    brand TEXT,
    eligible_for_model BOOLEAN,
    reconciled_forecast DOUBLE PRECISION,
    actual_ly DOUBLE PRECISION,
    actual_last DOUBLE PRECISION,
    growth_vs_ly_abs DOUBLE PRECISION,
    growth_vs_last_abs DOUBLE PRECISION,
    growth_vs_ly_pct DOUBLE PRECISION,
    growth_vs_last_pct DOUBLE PRECISION,
    national_growth_vs_ly_abs DOUBLE PRECISION,
    national_growth_vs_last_abs DOUBLE PRECISION,
    c2g_vs_ly DOUBLE PRECISION,
    c2g_vs_last DOUBLE PRECISION,
    driver_direction TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_prescription_actions" (
    ds DATE,
    level TEXT,
    nestle_region TEXT,
    nestle_store_cluster TEXT,
    store_code TEXT,
    product_code TEXT,
    recommendation TEXT,
    rationale TEXT,
    trend_signal TEXT,
    forecast_signal TEXT,
    c2g_signal TEXT
)
''',

f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."api_prescription_implementation_guide" (
    sort_order INTEGER,
    label TEXT,
    value TEXT
)
'''
]

for sql in api_sql:
    run_sql(sql)

print("API tables ready")


## 5) Deployment / app support tables

These are useful once the website triggers the pipeline.


In [ ]:

run_sql(f'''
CREATE TABLE IF NOT EXISTS "{SCHEMA}"."pipeline_runs" (
    run_id BIGSERIAL PRIMARY KEY,
    run_started_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    run_finished_at TIMESTAMP,
    status TEXT,
    triggered_by TEXT,
    source_file TEXT,
    notes TEXT
)
''')

print("pipeline_runs ready")


## 6) Optional cleanup of old legacy tables

Run this only if you are sure the old warehouse-only tables are no longer needed.


In [ ]:

# Uncomment only when ready to remove legacy tables no longer used by the current pipeline.

# legacy_tables = [
#     "fact_sales",
#     "fact_decomposition",
#     "fact_seasonal_profile",
#     "fact_forecast",
#     "fact_forecast_reconciled",
#     "fact_allocation",
# ]
#
# for t in legacy_tables:
#     run_sql(f'DROP TABLE IF EXISTS "{SCHEMA}"."{t}" CASCADE')
#
# print("Legacy tables removed")


## 7) Quick verification


In [ ]:

verify_sql = f'''
SELECT table_name
FROM information_schema.tables
WHERE table_schema = '{SCHEMA}'
ORDER BY table_name
'''
import pandas as pd
with engine.begin() as conn:
    tables_df = pd.read_sql_query(text(verify_sql), conn)

print(f"Total tables in {SCHEMA}: {len(tables_df)}")
tables_df


## Final note

This notebook is aligned to the **real uploaded original source files** you provided later in the conversation.

It now includes:
- the true stage handoff tables
- the true mart tables
- the true dashboard `api_*` tables
- optional cleanup for older unused tables

If you want, the next good step is to generate a second notebook called **`dq_validation_setup.ipynb`** that creates:
- null validation rules
- structural null rules by level
- freshness checks
- schema checks
- row count contract checks
for all of these tables.
